# 00 — Locations CSV first-pass inspection

Profiles the provided `data/raw/locations.csv` (~1M rows) with DuckDB — no full load into RAM.
Feeds the **data-quality issues** section of `docs/data-sources.md`.

Drop the provided file at `data/raw/locations.csv` before running (see `data/raw/README.md`).

In [ ]:
import duckdb
from leo_pipeline.config import PATHS

CSV = PATHS.locations_csv
assert CSV.exists(), f"Place the provided file at {CSV} (see data/raw/README.md)"
con = duckdb.connect()
rel = f"read_csv_auto('{CSV}', sample_size=-1)"
print(CSV)

In [ ]:
# Schema + row count
print(con.sql(f"DESCRIBE SELECT * FROM {rel}"))
print(con.sql(f"SELECT count(*) AS n_rows FROM {rel}"))

In [ ]:
# Quality checks — adjust lat/lon column names to the real schema once known.
LAT, LON = 'latitude', 'longitude'
con.sql(f"""
SELECT
  count(*)                                                    AS total,
  count(*) FILTER (WHERE {LAT} IS NULL OR {LON} IS NULL)      AS null_coords,
  count(*) FILTER (WHERE {LAT} NOT BETWEEN -90 AND 90)        AS bad_lat,
  count(*) FILTER (WHERE {LON} NOT BETWEEN -180 AND 180)      AS bad_lon,
  count(*) FILTER (WHERE {LAT} = 0 AND {LON} = 0)             AS null_island
FROM {rel}
""")

In [ ]:
# Duplicate coordinates
con.sql(f"""
SELECT count(*) AS duplicate_coord_groups FROM (
  SELECT {LAT}, {LON} FROM {rel} GROUP BY 1, 2 HAVING count(*) > 1
)
""")